In [5]:
import pandas as pd
import pyodbc

# Đã thêm chữ 'r' để sửa lỗi SyntaxWarning
conn = pyodbc.connect('DRIVER={ODBC Driver 17 for SQL Server};'
                      r'SERVER=DESKTOP-4LA7R0I\SQLEXPRESS;'
                      'DATABASE=DAP_Campaign_Attribution;'
                      'Trusted_Connection=yes;')

query = "SELECT * FROM User_Attribution_Features"

# Tạo con trỏ (cursor) để chạy lệnh SQL
cursor = conn.cursor()
cursor.execute(query)

# Lấy toàn bộ dữ liệu (rows)
rows = cursor.fetchall()

# Lấy tên các cột trong bảng (columns)
columns = [column[0] for column in cursor.description]

# Đưa dữ liệu thô vào Pandas DataFrame
df = pd.DataFrame.from_records(rows, columns=columns)

# Đóng kết nối để giải phóng bộ nhớ
cursor.close()
conn.close()

print(df.head())

   User_ID  is_converted  touch_Social_Media  touch_Email  touch_Search_Ads  \
0    61019             0                   0            0                 0   
1    55446             1                   1            1                 1   
2    26727             1                   1            1                 1   
3    96900             1                   1            0                 0   
4    28387             1                   0            0                 1   

   touch_Display_Ads  touch_Direct_Traffic  
0                  1                     1  
1                  0                     1  
2                  0                     1  
3                  0                     0  
4                  1                     0  


In [9]:
import statsmodels.api as sm

# Tách biến độc lập (X) và biến mục tiêu (y)
# Đã sửa lại tên các cột cho khớp với dữ liệu thực tế:
X = df[['touch_Social_Media', 'touch_Email', 'touch_Search_Ads', 'touch_Display_Ads', 'touch_Direct_Traffic']] 

# Sửa lại tên cột mục tiêu thành chữ thường:
y = df['is_converted']

# Thêm hằng số (intercept) vào mô hình
X = sm.add_constant(X)

# Huấn luyện mô hình Logistic Regression
logit_model = sm.Logit(y, X)
result = logit_model.fit()

# In bảng tóm tắt thống kê (sẽ bao gồm cả Pseudo-R2)
print(result.summary())

Optimization terminated successfully.
         Current function value: 0.393416
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:           is_converted   No. Observations:                 2847
Model:                          Logit   Df Residuals:                     2841
Method:                           MLE   Df Model:                            5
Date:                Tue, 26 May 2026   Pseudo R-squ.:                  0.1174
Time:                        17:32:26   Log-Likelihood:                -1120.1
converged:                       True   LL-Null:                       -1269.0
Covariance Type:            nonrobust   LLR p-value:                 2.915e-62
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                   -0.1948      0.117     -1.659      0.097      -0.425       0.035

In [14]:
from sklearn.metrics import roc_auc_score

# 1. Lấy Pseudo-R2 (McFadden) trực tiếp từ kết quả của statsmodels
pseudo_r2 = result.prsquared
print(f"Pseudo-R2 (McFadden): {pseudo_r2:.4f}")

# 2. Tính toán AUC
# Dự đoán xác suất chuyển đổi
y_pred_prob = result.predict(X)

# Tính AUC
auc_score = roc_auc_score(y, y_pred_prob)
print(f"AUC: {auc_score:.4f}")

Pseudo-R2 (McFadden): 0.1174
AUC: 0.7438


In [15]:
import scipy.stats as stats

# Lấy hệ số của các kênh (bỏ qua hằng số 'const')
coefficients = result.params.drop('const')

# Xếp hạng các kênh dựa trên hệ số hồi quy (Logistic Ranking)
# Hệ số càng cao -> Xếp hạng càng cao (hạng 1, 2, 3...)
logistic_ranking = coefficients.rank(ascending=False)

# Giả sử bạn đã có list xếp hạng của mô hình Linear từ RQ1 
# (Ví dụ: Social hạng 1, Search hạng 2...)
linear_ranking = [1, 3, 2, 4, 5] # Đây chỉ là số liệu ví dụ 

# Tính hệ số tương quan hạng Spearman
rho, p_value = stats.spearmanr(logistic_ranking, linear_ranking)
print(f"Spearman's Correlation (Logistic vs Linear): {rho:.4f}")

Spearman's Correlation (Logistic vs Linear): -0.6000
